In [2]:
from scripts.utils import *
from scripts.setup_evcouplings import *
from scripts.get_structures_and_sequences import *
from scripts.structural_filter import *
from scripts.hmm_domain_filter import *

In [3]:
# IMPORTANT PARAMETERS TO SET BEFORE RUNNING:
# evcouplings, foldseek, USalign, hmmpress and hmmsearch should be installed and in Path

CPUS = 16
BIT_SCORES = [0.8]
SEED= ["Q5SK37", 507, 569] # [uid,region_start,region_end]
HMMs_DIR = "STAC_hmm_domains" # should contain GA cutoff
CONFIG_TEMPLATE = "evcouplings_config_template.yaml"

In [4]:
# directory setup for each run:

# root_dir
    # data
        # evcouplings
            # contains config file and output
        # filtering
            # usalgin_dir
            # foldseek_dir
            # structural_filtering_output.csv
            # True_Members.csv
            # structures_containing_seeds_and_not_al_domains.csv
            # hmm_domain_filtering_dir
        # alphafold_structures
            # contains structures of potential true members
        # seed
            # contains structure of seed
        # HMMs
            # contains HMMs
        # all_sequences_full.fasta

import time 

def setup_directories():
    root_dir = f"runs/true_members_run_{SEED[0]}_{time.strftime('%Y-%m-%d_%H-%M-%S')}"
    directories = [
        f"{root_dir}/data/evcouplings",
        f"{root_dir}/data/filtering/usalign",
        f"{root_dir}/data/filtering/foldseek",
        f"{root_dir}/data/filtering/hmm_domain_filtering",
        f"{root_dir}/data/alphafold_structures",
        f"{root_dir}/data/seed",
        f"{root_dir}/data/HMMs",
    ]
    for directory in directories:
        # clear directory if it already exists
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory, exist_ok=True)
    return root_dir
        
root_dir = setup_directories()

In [5]:
# Run evcouplings align and couplings stage
run_evcouplings(SEED, BIT_SCORES, CPUS, CONFIG_TEMPLATE, root_dir)

FileNotFoundError: [Errno 2] No such file or directory: 'evcouplings'

In [6]:
number_tracker = {} # dictionary to keep track of true members at each stage in the pipeline

In [7]:
# Get all structures and full fasta of all sequences of evcouplings output
EVCOUPLINGS_OUT_DIR = f"{root_dir}/data/evcouplings",
number_tracker = get_all_structures_and_sequences(SEED, BIT_SCORES, number_tracker, root_dir)

Could not extract UniProt accession from record ID: UPI000DF2BD3B
Could not extract UniProt accession from record ID: UPI0003824F5A
Could not extract UniProt accession from record ID: UPI001FAAB0B1
Could not extract UniProt accession from record ID: UPI000542879D
Could not extract UniProt accession from record ID: UPI00263A1D77
Could not extract UniProt accession from record ID: UPI00023898AB
Total unique uniprot UIDs for AlphaFold: 12/18

Skipping entry 'A0ABV6Q133': could not download AlphaFold structure.
Processing 2/12 structures...
Processing 4/12 structures...
Processing 6/12 structures...
Processing 8/12 structures...
Processing 10/12 structures...
Processing 12/12 structures...
Downloaded 11/12 structure files (1 failed).
Fetched 11/11 records ...


In [8]:
# Run USalgin, foldseek and filter output
number_tracker = run_structural_similarity_filter(SEED, number_tracker, root_dir)

Processing 1/1 structures...
Downloaded 1/1 structure files (0 failed).
Extracting region (507, 569) from protein file: runs/true_members_run_Q5SK37_2026-06-04_11-55-09/data/seed/Q5SK37.cif
Remaining structures after filtering of foldseek output: 10/10
Remaining structures after filtering usalign output: 10/11


In [9]:
# Run hmmsearch with CbrA related HMMs on fasta of STAC containing sequences and filter to get STAC containing CbrAs and STAC containing proteins without CbrA domains
seed_containing_sequences_ids_path = f"{root_dir}/data/filtering/structural_filtering_output_tm_0.6.csv"
number_tracker = run_hmm_domain_filter(number_tracker, CPUS, seed_containing_sequences_ids_path, HMMs_DIR, root_dir)

Found 10/10 matching records in the FASTA file


In [10]:
# Print number of true members at each stage in the pipeline
for key, value in number_tracker.items():
    print(f"{key}: {value}")

annotation_files: 1
unique_rep_ids: 18
unique_uniprot_uids: 12
successful_structure_downloads: 11
downloaded_records: 11
number_filtered_structures_foldseek: 10
number_filtered_structures_usalign: 10
number_filtered_structures_total: 10
structures_with_seed_and_domains: 0
structures_with_seed_and_not_all_domains: 10


In [10]:
# Feed results into STAC containing CbrA and STAC containing protein datasets and compute statistics 